# 3. Model Analysis

This notebook analyzes model predictions and investigates individual creatures.

**Input:**
- `helper_files/engineered_features.parquet`
- `pickled_models/hp_model_cr*.pkl`

**Output:**
- `data/engineered_features.csv`
- `data/feature_contributions.csv`

## Imports and Configs

In [6]:
import numpy as np
import os
import pandas as pd
import sys

from pathlib import Path

In [11]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IO_DIR = './notebooks_io'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IO_DIR = './notebooks/notebooks_io'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [12]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        get_phase3_features,
        get_cr_tier,
        PHASE2_PENALTIES,
        load_model,
        summarize_model_performance,
        add_percentile_by_cr,
        investigate_creature,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        get_phase3_features,
        get_cr_tier,
        PHASE2_PENALTIES,
        load_model,
        summarize_model_performance,
        add_percentile_by_cr,
        investigate_creature,
    )

    print("Imports successful")


Imports successful


## Load Data and Models

In [13]:
# Load engineered features
load_path = IO_DIR + "/engineered_features.parquet"
df = pd.read_parquet(load_path)
print(f"Loaded {len(df)} monsters")

Loaded 324 monsters


In [15]:
# Load models
models = {}
scalers = {}
os.makedirs(PICKLED_MODELS_DIR, exist_ok=True)
load_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    filepath = load_path.replace('tier', tier)
    print(filepath)
    data = load_model(filepath)
    models[tier] = data['model']
    scalers[tier] = data['scaler']
    print(f"Loaded {tier} model")

phase3_features = get_phase3_features()

./pickled_models/hp_model_cr1.pkl


KeyError: 'model'

In [17]:
data.keys()

dict_keys(['coef', 'intercept', 'scaler', 'feature_columns', 'phase2_penalties', 'cr_range', 'cr_label', 'test_r2', 'test_mae'])

## Generate Predictions

In [ ]:
def get_prediction_for_creature(row):
    """Get the final HP prediction for a creature based on its CR tier."""
    tier = row['cr_tier']
    
    # Get features
    X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
    
    # Scale and predict
    X_scaled = scalers[tier].transform(X)
    residual_pred = models[tier].predict(X_scaled)[0]
    
    return row['hp_after_phase2'] + residual_pred

df['predicted_hp'] = df.apply(get_prediction_for_creature, axis=1)
df['hp_delta'] = df['predicted_hp'] - df['actual_hp']
df['hp_delta_pct'] = (df['hp_delta'] / df['actual_hp']) * 100

df = add_percentile_by_cr(df)

print("Predictions generated")

In [ ]:
# Summary statistics
print("Overall Prediction Summary:")
print(f"   Mean HP Error: {df['hp_delta'].mean():.1f} HP")
print(f"   Mean Absolute Error: {df['hp_delta'].abs().mean():.1f} HP")
print(f"   Mean % Error: {df['hp_delta_pct'].mean():.1f}%")
print(f"   Mean Absolute % Error: {df['hp_delta_pct'].abs().mean():.1f}%")

In [ ]:
# # Train all models
# results = {}

# results['cr1'] = train_tier_model(train_cr1, test_cr1, 'CR < 1', phase3_features)
# results['cr2'] = train_tier_model(train_cr2, test_cr2, 'CR 1-4', phase3_features)
# results['cr3'] = train_tier_model(train_cr3, test_cr3, 'CR 5-10', phase3_features)
# results['cr4'] = train_tier_model(train_cr4, test_cr4, 'CR 11-16', phase3_features)
# results['cr5'] = train_tier_model(train_cr5, test_cr5, 'CR > 16', phase3_features)

# summarize_model_performance(results)

In [ ]:
# Summary by CR tier
tier_labels = {
    'cr1': 'CR < 1',
    'cr2': 'CR 1-4',
    'cr3': 'CR 5-10',
    'cr4': 'CR 11-16',
    'cr5': 'CR > 16',
}

print("\nBy CR Tier:")
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    tier_df = df[df['cr_tier'] == tier]
    mae = tier_df['hp_delta'].abs().mean()
    mape = tier_df['hp_delta_pct'].abs().mean()
    print(f"   {tier_labels[tier]:12s}: MAE={mae:6.1f} HP, MAPE={mape:5.1f}%")

## Basic 

## Feature Contributions

In [ ]:
def calculate_feature_contributions(row):
    """Calculate HP contribution of each feature for a creature."""
    tier = row['cr_tier']
    penalties = PHASE2_PENALTIES[tier]
    
    contributions = {
        'Name': row['Name'],
        'CR': row['cr_numeric'],
        'actual_hp': row['actual_hp'],
        'predicted_hp': row['predicted_hp'],
        'hp_error': row['hp_delta'],
        'hp_error_pct': row['hp_delta_pct'],
        'hp_baseline': row['hp_baseline'],
        'hp_after_phase1_5': row['hp_after_phase1_5'],
        'hp_after_phase2': row['hp_after_phase2'],
        'phase1_5_resistance_penalty': row.get('resistance_penalty', 0),
        'phase1_5_immunity_penalty': row.get('immunity_penalty', 0),
        'phase1_5_total_penalty': row.get('total_defensive_penalty', 0),
    }
    
    # Phase 2 contributions
    contributions['phase2_ac_contribution'] = row['ac_deviation'] * penalties.get('ac_deviation', 0)
    contributions['phase2_attack_contribution'] = row['attack_deviation'] * penalties.get('attack_deviation', 0)
    contributions['phase2_dpr_contribution'] = row['dpr_deviation'] * penalties.get('dpr_deviation', 0)
    contributions['phase2_save_dc_contribution'] = row['save_dc_deviation'] * penalties.get('save_dc_deviation', 0)
    contributions['phase2_flying_contribution'] = row['has_flying'] * penalties.get('has_flying', 0)
    contributions['phase2_advantage_contribution'] = row.get('has_advantage_condition', 0) * penalties.get('has_advantage_condition', 0)
    contributions['phase2_disadvantage_contribution'] = row.get('has_disadvantage_condition', 0) * penalties.get('has_disadvantage_condition', 0)
    contributions['phase2_attackers_advantage_contribution'] = row.get('has_attackers_advantage', 0) * penalties.get('has_attackers_advantage', 0)
    contributions['phase2_prone_contribution'] = row.get('inflicts_prone', 0) * penalties.get('inflicts_prone', 0)
    
    contributions['phase2_total_contribution'] = sum([
        contributions['phase2_ac_contribution'],
        contributions['phase2_attack_contribution'],
        contributions['phase2_dpr_contribution'],
        contributions['phase2_save_dc_contribution'],
        contributions['phase2_flying_contribution'],
        contributions['phase2_advantage_contribution'],
        contributions['phase2_disadvantage_contribution'],
        contributions['phase2_attackers_advantage_contribution'],
        contributions['phase2_prone_contribution'],
    ])
    
    # Phase 3 contributions
    X = np.array([[row.get(f, 0) for f in phase3_features]])
    X = np.nan_to_num(X, 0)
    X_scaled = scalers[tier].transform(X)[0]
    coefs = models[tier].coef_
    
    phase3_total = 0
    for i, feature in enumerate(phase3_features):
        contrib = X_scaled[i] * coefs[i]
        contributions[f'phase3_{feature}'] = contrib
        phase3_total += contrib
    
    contributions['phase3_intercept'] = models[tier].intercept_
    contributions['phase3_total_contribution'] = phase3_total + contributions['phase3_intercept']
    
    return pd.Series(contributions)

# Calculate contributions for all creatures
contributions_df = df.apply(calculate_feature_contributions, axis=1)
print(f"Calculated contributions for {len(contributions_df)} creatures")

## Export Data

In [ ]:
# Export engineered features
export_columns = [
    'Name', 'Type', 'Size', 'Challenge_Rating', 'cr_numeric', 'cr_tier',
    'HP', 'actual_hp', 'AC', 'ac_value',
    'predicted_hp', 'hp_delta', 'hp_delta_pct', 'hp_delta_pct_percentile',
    'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline', 'dc_baseline',
    'highest_attack_bonus', 'highest_save_dc', 'estimated_dpr', 'legendary_dpr', 'total_dpr',
    'ac_deviation', 'attack_deviation', 'dpr_deviation', 'save_dc_deviation',
    'hp_after_phase1_5', 'hp_after_phase2', 'residual_hp',
]

# Add all existing columns that match
export_cols = [c for c in export_columns if c in df.columns]

export_df = df[export_cols].sort_values('cr_numeric')

export_path = DATA_DIR + '/engineered_features.csv'
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df)} monsters to data/engineered_features.csv")

In [ ]:
# Export contributions
export_path = DATA_DIR + '/feature_contributions.csv'
contributions_df.to_csv(export_path, index=False)
print(f"Exported contributions to data/feature_contributions.csv")

## Investigate Specific Creatures

In [ ]:
# Example: Investigate a creature
investigate_creature('Elephant', df, contributions_df)

In [ ]:
# List worst predictions
print("\nWorst Over-predictions (predicted > actual):")
over_pred = df[df['hp_delta'] > 0].nlargest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(over_pred.to_string(index=False))

In [ ]:
print("\nWorst Under-predictions (predicted < actual):")
under_pred = df[df['hp_delta'] < 0].nsmallest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(under_pred.to_string(index=False))

In [ ]:
print("\nBest Predictions:")
best = df.nsmallest(10, df['hp_delta_pct'].abs())[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(best.to_string(index=False))

In [ ]:
# Interactive investigation
# Uncomment and modify to investigate specific creatures:
# investigate_creature('Adult Red Dragon', df, contributions_df)
# investigate_creature('Tarrasque', df, contributions_df)
investigate_creature('Goblin', df, contributions_df)

# Deatiled Model Breakdown